# State and validity

`PyBaMMOracle` is a stateful DFN cell: each call runs one charge/discharge cycle that
continues from the cell the previous calls left, and returns the impedance features an
experiment would measure. This notebook shows:

1. building an oracle and running cycles;
2. what carries from one call to the next (degradation, state of health);
3. `reset()` and the seed;
4. checking that a solution is physically valid (`battery_oracle.validity`).

The Randles-stub ECM is used so the notebook runs without the `[autoeis]` extra. Each
cycle is a full DFN solve (tens of seconds).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from battery_oracle import Protocol, PyBaMMOracle, randles_stub_ecm
from battery_oracle import validity

## 1. An oracle and one cycle

A protocol is a two-stage constant-current charge and a constant-current discharge
(durations in h). Currents are in mA of the real cell the experiment runs on
(`real_cell_capacity_mah`, 200 mAh by default); the oracle scales them to its own model
cell and clamps them to its bounds (each charge stage at most 15 min). `run_cycle` returns a
`CycleResult`; calling the oracle directly returns only its state vector.

In [ ]:
oracle = PyBaMMOracle(ecm_model_fn=randles_stub_ecm, seed=0)
protocol = Protocol(charge_current_1_mA=100.0, charge_current_2_mA=50.0,      # C/2, then C/4
                    charge_duration_1_h=0.25, charge_duration_2_h=0.25,  # <= 15 min per stage
                    discharge_current_mA=100.0, discharge_duration_h=2.0)
result = oracle.run_cycle(protocol)
print("SOH", round(result.soh, 6), "| capacity", round(result.capacity_ah, 4), "Ah | solver status", result.failure_kind)
print("state vector", result.state.shape, "blocks", list(result.schema))
print("charged-state ECM", {k: round(v, 5) for k, v in result.ecm_charge.items()})

## 2. State carries across calls

The next call starts from the last call's solution, so the lithium lost to side
reactions accumulates and the state of health only falls. Alternate a gentle and a
harsh protocol for a few cycles.

In [ ]:
gentle = protocol.to_array()
harsh = gentle.copy()
harsh[[0, 1, 4]] *= 2.0
for k in range(4):
    oracle(harsh if k % 2 else gentle)

h = oracle.history
fig, ax = plt.subplots(1, 2, figsize=(9, 3.4))
ax[0].plot([r["call_idx"] for r in h], [r["end_soh"] for r in h], "o-")
ax[0].set_xlabel("Call"); ax[0].set_ylabel("State of health (-)")
ax[1].plot([r["call_idx"] for r in h], [1e3 * r["cumulative_lli_total_ah"] for r in h], "s-")
ax[1].set_xlabel("Call"); ax[1].set_ylabel("Lithium lost (mAh)")
for a in ax: a.grid(alpha=0.3)
fig.tight_layout()

## 3. `reset()` and the seed

`reset()` returns to the fresh cell and re-seeds the measurement-noise generator, so the
same calls reproduce the same trajectory, noise included. A different seed changes the
noise on the spectra, not the cell.

In [ ]:
first = [r["end_soh"] for r in oracle.history[:2]]
oracle.reset()
again = [oracle.run_cycle(p).soh for p in (gentle, gentle)]
print("before reset", first)
print("after reset ", again)

other = PyBaMMOracle(ecm_model_fn=randles_stub_ecm, seed=1)
other(gentle)
z0, z1 = oracle.history[0]["Z_charge_real"], other.history[0]["Z_charge_real"]
print("same SOH:", np.isclose(other.history[0]["end_soh"], oracle.history[0]["end_soh"]),
      "| spectra differ by noise:", not np.allclose(z0, z1))

## 4. Is the solution physical?

`validity.check_solution` judges one solved history on laws the DFN must satisfy: the
electrolyte current, flux, concentration and potential are continuous across the
electrode|separator interfaces, the electrolyte and solid currents add up to the applied
current, no current leaves through the collectors, the mesh does not oscillate, and
lithium is conserved. Each record has a status (`pass`, `warn`, `fail`, `skip`), the
value and its limits.

In [ ]:
for c in validity.check_solution(oracle.last_solution, oracle.last_parameter_values):
    print(f"{c['status']:5s} {c['name']:60s} {c['value']}")

A single solution can pass these checks and still be under-resolved. `validity.judge`
compares the measures of the same history on two meshes (and at tighter solver
tolerances for the exact laws), and `battery_oracle.physicality.check_dfn_cycle` runs that
judgement on one demanding cycle of a parameterised oracle; the stage-A fit uses both.